# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

Command line and Python basics (01):
In Lec01, we learned about len(items) being used to count items.
For example in Task 5.1: "row_check": len(analysis) == len(visits_clean). This was to make sure the items in analysis equaled the items in visits_clean



Git, functions, and files (02):
In Lec02, we learned how to create a dictionary, which was utilizied heavily in this assignment.
For example: in 1.2: return {
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "exact_duplicate_rows": table.duplicated().sum()
    }.
    


Jupyter and labeled pandas data (04)
Lec04 is where we learned how to build series and dataframes, also utilized quite a bit here.
For example in Task 3.1: summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max", "site_code values", "patients at N01", "patients at S02", "patients at E04"],
    "value": [len(patients_clean), patients_clean["birth_year"].isna().sum(), patients_clean["birth_year"].min(), patients_clean["birth_year"].max(), patients_clean["site_code"].nunique(),
              (patients_clean["site_code"] == "N01").sum(),
              (patients_clean["site_code"] == "S02").sum(),
              (patients_clean["site_code"] == "E04").sum()
    ]
})


Data contracts, cleaning, and validation (05):
In Lec05, we learned about validation checks, and how to use assert checks.all(), checks[~[checks], which stops with AssertionError when the assert fails and lists failed rules, with nothing running after. 
For example, it's used in Task 5.1: assert checks.all(), checks[~checks]


Joins, reshaping, and concatenation (06):
Lec06 we learned about concatenation, and stacking rows with concat, like using pd.concat([df1, df2]) to stack tables with the same columns.

For example, used in Task 2.2: visits_raw = pd.concat(
    [raw["visits_north"].copy(deep=True).assign(source_file="visits_north.csv"),
     raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"}).assign(source_file="visits_south.csv")],
    ignore_index=True
    
)

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash

wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:


def profile(name, table):
    return {
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "exact_duplicate_rows": table.duplicated().sum()
    }

results = []
for name in manifest["files"]:
    table = raw[Path(name).stem]
    results.append(profile(name, table))

profile_df = pd.DataFrame(results)
profile_df.to_csv(PROFILE_PATH, index=False)


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

### 2.2 Stack the two visit files

In [4]:

visits_raw = pd.concat(
    [raw["visits_north"].copy(deep=True).assign(source_file="visits_north.csv"),
     raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"}).assign(source_file="visits_south.csv")],
    ignore_index=True
    
)

### 2.3 Audit the raw tables

In [5]:

site_code_clean = (raw["patients"]["site_code"].fillna("").str.upper().str.replace(" ", "").str.replace("-", ""))

dates = pd.to_datetime(visits_raw["visit_date"],
    format= "%Y-%m-%d",
    errors = "coerce"
)

visit_map = {
    "baseline": "baseline", "week12": "week12", "bl": "baseline", "wk12": "week12", "week 12": "week12"
    
}


visit_labels = (
    visits_raw["visit"]
    .str.strip()
    .str.lower()
    .map(visit_map)
)

repeated_pairs = visits_raw.copy()
repeated_pairs["visit"] = visit_labels

sbp = pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce")

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs"
    ],
    "count": [
        ((raw["patients"]["gender"].fillna("").str.strip() != "") &
        ~raw["patients"]["gender"].isin(["F", "M", "X"])).sum(),
        raw["patients"]["gender"].fillna("").str.strip().eq("").sum(),
        
        pd.to_numeric(raw["patients"]["birth_year"], errors="coerce").eq(9999).sum(),

    
        (raw["patients"]["site_code"].fillna("") != site_code_clean).sum(),

        raw["sites"].duplicated(subset=["site_code"], keep=False).sum(),
        (~visits_raw["visit"].isin(["baseline", "week12"])).sum(),
        pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce").eq(999).sum(),

        ((pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") != 999) &
        ((pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") < 70) |
        (pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") > 250))).sum(),
        dates.isna().sum(),
        repeated_pairs.duplicated(subset=["patient_id", "visit"], keep=False).sum()
    ]
}
)

issue_audit.to_csv(AUDIT_PATH, index=False)
assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."
print(issue_audit.shape)
print(issue_audit)

(10, 2)
                                               issue  count
0        patients: gender values that are not a code   6968
1                             patients: empty gender    486
2                 patients: birth_year sentinel 9999    371
3         patients: site codes needing normalization   1811
4              sites: rows with a repeated site_code      4
5  visits: visit labels that are not baseline or ...   2309
6                           visits: sbp sentinel 999    341
7                 visits: sbp outside 70 through 250    116
8                        visits: date parse failures    127
9  visits: rows in repeated patient_id and visit ...    380


## Task 3: Clean

### 3.1 Clean the patients

In [6]:

raw["patients"]["gender"].value_counts(dropna=False)
patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates()
patients_clean["gender"] = patients_clean["gender"].str.strip().str.lower().map({"female": "F", "woman": "F", "f": "F", "male": "M", "m": "M", "man": "M", "other": "X", "x": "X"})
patients_clean["gender"] = patients_clean["gender"].astype("category")
patients_clean["birth_year"] = patients_clean["birth_year"].replace("9999", pd.NA).astype("Int64")
patients_clean["site_code"] = patients_clean["site_code"].str.replace("-", "").str.replace(" ", "").str.upper()
patients_clean = patients_clean.reset_index(drop=True)

#nunique covered?
summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max", "site_code values", "patients at N01", "patients at S02", "patients at E04"],
    "value": [len(patients_clean), patients_clean["birth_year"].isna().sum(), patients_clean["birth_year"].min(), patients_clean["birth_year"].max(), patients_clean["site_code"].nunique(),
              (patients_clean["site_code"] == "N01").sum(),
              (patients_clean["site_code"] == "S02").sum(),
              (patients_clean["site_code"] == "E04").sum()
    ]
})


In [7]:
# TODO: show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.
patients_clean["gender"].cat.categories
summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)
gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)


### 3.2 Clean the visits

In [8]:

visits_raw["visit"].value_counts(dropna=False)
visits_clean = visits_raw.copy(deep=True)
    
visits_clean = visits_clean.drop_duplicates()
#subset?
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower()
visits_clean["visit"] = visits_clean["visit"].map({"baseline": "baseline", "bl": "baseline", "week12": "week12", "wk12": "week12", "week 12": "week12", "week12": "week12"})
visits_clean["sbp_mmhg"] = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce")
visits_clean["sbp_mmhg"] = visits_clean["sbp_mmhg"].where((visits_clean["sbp_mmhg"] >= 70) & (visits_clean["sbp_mmhg"] <= 250))
visits_clean["sbp_mmhg"] = visits_clean["sbp_mmhg"].astype("Int64")
visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")
visits_clean["visit_date"] = pd.to_datetime(visits_clean["visit_date"], format="%Y-%m-%d", errors="coerce")


In [9]:

visits_clean[visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)]
visits_clean = visits_clean.sort_values("visit_time").drop_duplicates(subset=["patient_id", "visit"], keep="last")


In [10]:


visits_clean = visits_clean.sort_values(["patient_id", "visit"])

visits_clean = visits_clean.reset_index(drop=True)

visits_clean = visits_clean[
    [
        "patient_id",
        "visit",
        "visit_date",
        "sbp_mmhg",
        "weight_kg",
        "source_file"
    ]
]

visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv"
    ],
    "value": [
        len(visits_clean),
        (visits_clean["visit"] == "baseline").sum(),
        (visits_clean["visit"] == "week12").sum(),
        visits_clean["visit_date"].isna().sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        visits_clean["sbp_mmhg"].isna().sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        visits_clean["weight_kg"].isna().sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        (visits_clean["source_file"] == "visits_north.csv").sum(),
        (visits_clean["source_file"] == "visits_south.csv").sum()
    ]
})


visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)


### 3.3 Record the decisions

In [11]:

decision_table = pd.DataFrame({
    "table": ["patients", "patients", "patients", "patients", "visits", "visits", "visits", "visits", "sites"],
    "issue": ["exact duplicate rows", "gender spellings that are not a code", "site codes needing normalization", "birth_year sentinel 9999", "weight column named wt_kg in visits_south.csv", "sbp sentinel 999 and readings outside 70 through 250", "impossible and loosely written visit dates", "repeated patient_id and visit pairs", "repeated site_code rows"],
    "decision": ["Removes exact duplicate rows, keeps the first", "List the spellings, strip and lowercase, and then map to codes for gender", "Site codes normalized when uppercased with every space and hyphen removed", "9999 sentinel becomes missing", "Weight column renamed to weight_kg", "Sentinel 999 becomes missing, and keep all readings within 70 through 250", "Oraganize the loosely written visits dates to a YYYY-MM-DD format", "Keep the later reading for repeated patient_id and visit pairs", "Repeated site_code rows are removed, keeping the first occurrence"],
    "reason": ["Duplicate rows may not be useful for analysis", "It becomes easier to make gender codes standardized", "Consistency is important for site codes for accuracy", "Sentinel 9999 means it is missing data", "Making the column names more consistent", "Sentinel 999 means it is missing data, and any readings outside 70-250 are outliers", "Loosely written visit dates need to organized into a standardized format to be easier to read", "Patient_id and visit pairs repeats are duplicate entries, which are unnecessary", "Duplicate site_code rows are also unnecessary"]
})

decision_table.to_csv(DECISION_LOG_PATH, index=False)


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [12]:

a1c_long = pd.melt(raw["a1c_wide"], id_vars=["patient_id"], value_vars=["a1c_baseline", "a1c_week12"], var_name="visit", value_name="a1c_pct")
a1c_long["visit"] = a1c_long["visit"].str.strip().str.lower().map({"a1c_baseline": "baseline", "a1c_week12": "week12"})
a1c_long["a1c_pct"] = pd.to_numeric(a1c_long["a1c_pct"], errors="coerce")
a1c_long["a1c_pct"].dtype

a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)
a1c_summary = pd.DataFrame({
    "visit": ["baseline", "week12"],
    "results": [
        (a1c_long["visit"] == "baseline").sum(),
        (a1c_long["visit"] == "week12").sum()
    ],

    "a1c_pct_mean": [
         round(a1c_long.loc[
            a1c_long["visit"] == "baseline", "a1c_pct"
        ].mean(), 2),
        round(a1c_long.loc[
            a1c_long["visit"] == "week12", "a1c_pct"
        ].mean(), 2)
    ]
})


a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)


### 4.2 Check the keys before joining

In [13]:

sites = raw["sites"] 
try:
    patients_sites = patients_clean.merge(
        sites, on = "site_code", how = "left", validate = "many_to_one")
except pd.errors.MergeError as error:
    print(error)
repeated_sites = sites[sites.duplicated(subset=["site_code"], keep=False)]
print(repeated_sites)
sites_current = sites.loc[
    sites["record_status"] == "current",
    ["site_code", "site_name"]
]

Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...
  site_code            site_name record_status
0       N01         North Clinic       current
1       N01  North Street Clinic       retired
2       S02     Southgate Clinic       retired
3       S02  South Valley Clinic       current


In [14]:


visit_a1c = visits_clean.merge(
    a1c_long,
    on=["patient_id", "visit"],
    how="outer",
    indicator=True
)
visit_a1c["_merge"].value_counts()

merge_audit = pd.DataFrame({
    "check": [
        "visits with an A1c result",
        "visits without an A1c result",
        "A1c results without a visit",
        "visits whose patient is not in the registry",
        "registry patients with no visit",
        "registry patients whose site is not in the site list",
        "site list rows removed by the current-record rule"
    ],
    "count": [
        (visit_a1c["_merge"] == "both").sum(),
        (visit_a1c["_merge"] == "left_only").sum(),
        (visit_a1c["_merge"] == "right_only").sum(),
        (~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum(),
        (~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum(),
        (~patients_clean["site_code"].isin(sites_current["site_code"])).sum(),
        len(sites) - len(sites_current)

    ]
})
merge_audit.to_csv(MERGE_AUDIT_PATH, index =False)



### 4.3 Pivot blood pressure to one row per patient

In [ ]:

sbp_wide = visits_clean.pivot(index = "patient_id", columns="visit", values="sbp_mmhg").reset_index()
    
sbp_wide.columns.name = None

sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]
sbp_wide_sample = sbp_wide[
        sbp_wide["patient_id"].isin(SAMPLE_IDS)
]


sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index = False)



#### NumPy checkpoint (Lecture 03, human review)

Environments, shell pipelines, and NumPy (03):
I chose axis=0 because choosing axis=1 means it would check across each row, while axis=0 checks each column, and we are trying to check the mean across patients for each of their visits.



In [ ]:



complete_pairs = sbp_wide[["baseline", "week12"]].dropna()

complete_pairs_array = np.array(complete_pairs, dtype = float)

print(complete_pairs_array.shape)
print(np.mean(complete_pairs_array, axis= 0)) 

(27149, 2)
[143.01337066 137.15061328]


### 4.4 Join the analysis table

In [ ]:

analysis = visits_clean.merge(
    patients_clean[["patient_id", "gender", "birth_year", "site_code"]], on = "patient_id", how = "left", validate = "many_to_one")

analysis = analysis.merge(
    sites_current, on = "site_code", how = "left", validate = "many_to_one")

analysis = analysis.merge(
    a1c_long, on = ["patient_id", "visit"], how = "left", validate = "one_to_one")

analysis = analysis[[
    "patient_id",
    "visit",
    "visit_date",
    "sbp_mmhg",
    "weight_kg",
    "a1c_pct",
    "gender",
    "birth_year",
    "site_code",
    "site_name",
    "source_file"
]]

analysis = analysis.astype({
    "patient_id": "string",
    "visit": "string",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string"
})

analysis["visit_date"] = analysis["visit_date"].astype("datetime64[us]") #check if needed





## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [ ]:


checks = pd.Series({
    "row_check": len(analysis) == len(visits_clean),
    "ID_visit_repeat": ~analysis.duplicated(subset=["patient_id", "visit"]).any(),
    "range": analysis["sbp_mmhg"].dropna().between(70, 250).all(),
    "current_site_name": analysis["site_name"].dropna().isin(sites_current["site_name"]).all(),
    "sbp_wide_unique": sbp_wide["patient_id"].is_unique,
    "raw_table_snap": all(raw[name].equals(raw_snapshot[name]) for name in raw)

})
print(checks)
assert checks.all(), checks[~checks]

row_check            True
ID_visit_repeat      True
range                True
current_site_name    True
sbp_wide_unique      True
raw_table_snap       True
dtype: bool


### 5.2 Save the analysis extract

In [ ]:



analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)

analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index = False)


### 5.3 Read the file back

In [ ]:

round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string"
    },
    parse_dates=["visit_date"]
        
)

print(round_trip.equals(analysis_sample))

True
